In [17]:

from collections import defaultdict
from dataclasses import dataclass
import os
import random
import time
from typing import Optional

import tqdm
from collections import defaultdict
from torch.utils.data import TensorDataset, DataLoader
from mani_skill.utils import gym_utils
from mani_skill.utils.wrappers.flatten import FlattenActionSpaceWrapper
from mani_skill.utils.wrappers.record import RecordEpisode
from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import tyro

import mani_skill.envs

In [2]:
# ALGO LOGIC: initialize agent here:
class SoftQNetwork(nn.Module):
    def __init__(self, env):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(np.array(env.single_observation_space.shape).prod() + np.prod(env.single_action_space.shape), 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, x, a):
        x = torch.cat([x, a], 1)
        return self.net(x)


LOG_STD_MAX = 2
LOG_STD_MIN = -5


class Actor(nn.Module):
    def __init__(self, env):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(np.array(env.single_observation_space.shape).prod(), 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )
        self.fc_mean = nn.Linear(256, np.prod(env.single_action_space.shape))
        self.fc_logstd = nn.Linear(256, np.prod(env.single_action_space.shape))
        # action rescaling
        h, l = env.single_action_space.high, env.single_action_space.low
        self.register_buffer("action_scale", torch.tensor((h - l) / 2.0, dtype=torch.float32))
        self.register_buffer("action_bias", torch.tensor((h + l) / 2.0, dtype=torch.float32))
        # will be saved in the state_dict

    def forward(self, x):
        x = self.backbone(x)
        mean = self.fc_mean(x)
        log_std = self.fc_logstd(x)
        log_std = torch.tanh(log_std)
        log_std = LOG_STD_MIN + 0.5 * (LOG_STD_MAX - LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std

    def get_eval_action(self, x):
        x = self.backbone(x)
        mean = self.fc_mean(x)
        action = torch.tanh(mean) * self.action_scale + self.action_bias
        return action

    def get_action(self, x):
        mean, log_std = self(x)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()  # for reparameterization trick (mean + std * N(0,1))
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        log_prob = normal.log_prob(x_t)
        # Enforcing Action Bound
        log_prob -= torch.log(self.action_scale * (1 - y_t.pow(2)) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)
        mean = torch.tanh(mean) * self.action_scale + self.action_bias
        return action, log_prob, mean

    def to(self, device):
        self.action_scale = self.action_scale.to(device)
        self.action_bias = self.action_bias.to(device)
        return super().to(device)

In [3]:
env_kwargs = dict(obs_mode="state", render_mode="rgb_array", sim_backend="gpu")
env_kwargs["control_mode"] = "pd_ee_delta_pos"


eval_envs = gym.make('PickCube-v1', num_envs=10, reconfiguration_freq=1, human_render_camera_configs=dict(shader_pack="default"), **env_kwargs)
eval_envs = ManiSkillVectorEnv(eval_envs, 10, ignore_terminations=True, record_metrics=True)

/opt/conda/lib/python3.9/site-packages/torch/random.py:187: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  warnings.warn(message)


In [31]:
@dataclass
class ReplayBufferSample:
    obs: torch.Tensor
    next_obs: torch.Tensor
    actions: torch.Tensor
    rewards: torch.Tensor
    dones: torch.Tensor
class ReplayBuffer:
    def __init__(self, env, num_envs: int, buffer_size: int, storage_device: torch.device, sample_device: torch.device):
        self.buffer_size = buffer_size
        self.pos = 0
        self.full = False
        self.num_envs = num_envs
        self.storage_device = storage_device
        self.sample_device = sample_device
        self.per_env_buffer_size = buffer_size // num_envs
        self.obs = torch.zeros((self.per_env_buffer_size, self.num_envs) + env.single_observation_space.shape).to(storage_device)
        self.next_obs = torch.zeros((self.per_env_buffer_size, self.num_envs) + env.single_observation_space.shape).to(storage_device)
        self.actions = torch.zeros((self.per_env_buffer_size, self.num_envs) + env.single_action_space.shape).to(storage_device)
        self.logprobs = torch.zeros((self.per_env_buffer_size, self.num_envs)).to(storage_device)
        self.rewards = torch.zeros((self.per_env_buffer_size, self.num_envs)).to(storage_device)
        self.dones = torch.zeros((self.per_env_buffer_size, self.num_envs)).to(storage_device)
        self.values = torch.zeros((self.per_env_buffer_size, self.num_envs)).to(storage_device)
        
        self.associated_r = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        self.Q = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        self.V = torch.zeros((self.per_env_buffer_size, num_envs), device=storage_device)
        
    def add_associated_reward(self, positions, rewards):
        rewards = rewards.unsqueeze(0).repeat(len(positions), 1)   #n_e -> len(positions), n_e
        self.associated_r[[positions]] += rewards    
        
    def add_qv_estimates(self, positions: list[int]):
        """
        Использует associated_r и rewards для вычисления Q(s,a) и V(s)
        """
        pos = torch.tensor(positions)  # [T]
        env_ids = torch.arange(self.num_envs)  # [N]
        ti, ei = torch.meshgrid(pos, env_ids, indexing="ij")  # [T, N]

        # [T, N]
        rewards = self.rewards[ti, ei]
        associated_r = self.associated_r[ti, ei]

        # Cumulative sum по временной оси (dim=0), но без включения текущего шага для Q
        # → Q = R - cumsum без текущего
        # → V = R - cumsum включая текущий
        cum_rewards_exclusive = torch.cumsum(rewards, dim=0) - rewards  # [T, N]
        cum_rewards_inclusive = torch.cumsum(rewards, dim=0)  # [T, N]

        q_values = associated_r - cum_rewards_exclusive
        v_values = associated_r - cum_rewards_inclusive

        self.Q[ti, ei] = q_values
        self.V[ti, ei] = v_values
        
    def sample_for_trans(self, positions: list):    
        positions = torch.tensor(positions)  # T
        env_ids = torch.arange(self.num_envs)  # N
        ti, ei = torch.meshgrid(positions, env_ids, indexing="ij")  # T, N

        obs_batch = self.obs[ti, ei]
        next_obs_batch = self.next_obs[ti, ei]
        actions_batch = self.actions[ti, ei]
        rewards_batch = self.rewards[ti, ei]
        dones_batch = self.dones[ti, ei]

        obs_batch = {k: v.to(self.sample_device) for k, v in obs_batch.items()}
        next_obs_batch = {k: v.to(self.sample_device) for k, v in next_obs_batch.items()}

        return ReplayBufferSample(
            obs=obs_batch,
            next_obs=next_obs_batch,
            actions=actions_batch.to(self.sample_device),
            rewards=rewards_batch.to(self.sample_device),
            dones=dones_batch.to(self.sample_device),
        )        

    def add(self, obs: torch.Tensor, next_obs: torch.Tensor, action: torch.Tensor, reward: torch.Tensor, done: torch.Tensor):
        if self.storage_device == torch.device("cpu"):
            obs = obs.cpu()
            next_obs = next_obs.cpu()
            action = action.cpu()
            reward = reward.cpu()
            done = done.cpu()

        self.obs[self.pos] = obs
        self.next_obs[self.pos] = next_obs

        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.dones[self.pos] = done

        self.pos += 1
        if self.pos == self.per_env_buffer_size:
            self.full = True
            self.pos = 0
    def sample(self, batch_size: int):
        if self.full:
            batch_inds = torch.randint(0, self.per_env_buffer_size, size=(batch_size, ))
        else:
            batch_inds = torch.randint(0, self.pos, size=(batch_size, ))
        env_inds = torch.randint(0, self.num_envs, size=(batch_size, ))
        return ReplayBufferSample(
            obs=self.obs[batch_inds, env_inds].to(self.sample_device),
            next_obs=self.next_obs[batch_inds, env_inds].to(self.sample_device),
            actions=self.actions[batch_inds, env_inds].to(self.sample_device),
            rewards=self.rewards[batch_inds, env_inds].to(self.sample_device),
            dones=self.dones[batch_inds, env_inds].to(self.sample_device)
        )
        
    def make_sequential_dataloader(self, positions: list[int], context_len: int, batch_size: int, shuffle: bool = True):
        pos = torch.tensor(positions)
        num_steps = len(pos)
        n_envs = self.num_envs
        device = self.sample_device   
        
        time_idx, env_idx = torch.meshgrid(pos, torch.arange(n_envs), indexing="ij")
        
        obs = self.obs[time_idx, env_idx]
        next_obs = self.next_obs[time_idx, env_idx]
        actions = self.actions[time_idx, env_idx]
        rewards = self.rewards[time_idx, env_idx]
        dones = self.dones[time_idx, env_idx]
        associated_r = self.associated_r[time_idx, env_idx]
        q_vals = self.Q[time_idx, env_idx]
        v_vals = self.V[time_idx, env_idx]
        
        sequences = []
        for start in range(num_steps - context_len + 1):
            end = start + context_len

            # Собираем последовательности: [context_len, n_envs, ...] → [n_envs, context_len, ...]
            obs_seq = obs[start:end].to(device).permute(1, 0, *range(2, obs.ndim))
            next_obs_seq = next_obs[start:end].to(device).permute(1, 0, *range(2, next_obs.ndim))
            actions_seq = actions[start:end].to(device).permute(1, 0, *range(2, actions.ndim))
            rewards_seq = rewards[start:end].to(device).permute(1, 0)
            dones_seq = dones[start:end].to(device).permute(1, 0)
            associated_r_seq = associated_r[start:end].to(device).permute(1, 0)
            q_seq = q_vals[start:end].to(device).permute(1, 0)
            v_seq = v_vals[start:end].to(device).permute(1, 0)

            for i in range(n_envs):
                seq = (
                    obs_seq[i],           # [context, ...]
                    next_obs_seq[i],
                    actions_seq[i],
                    rewards_seq[i],
                    dones_seq[i],
                    associated_r_seq[i],
                    q_seq[i],
                    v_seq[i]
                )
                sequences.append(seq)
                
        obs_seq = torch.stack([s[0] for s in sequences])
        next_obs_seq = torch.stack([s[1] for s in sequences])
        actions_seq = torch.stack([s[2] for s in sequences])
        rewards_seq = torch.stack([s[3] for s in sequences])
        dones_seq = torch.stack([s[4] for s in sequences])
        associated_r_seq = torch.stack([s[5] for s in sequences]) 
        q_seq = torch.stack([s[6] for s in sequences])
        v_seq = torch.stack([s[7] for s in sequences])     
        
        dataset = TensorDataset(
            *list(obs_seq.values()),
            *list(next_obs_seq.values()),
            actions_seq,
            rewards_seq,
            dones_seq,
            associated_r_seq,
            q_seq,
            v_seq
        )
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

        return dataloader    

In [6]:
rb = ReplayBuffer(
        env=eval_envs,
        num_envs=100,
        buffer_size=100000,
        storage_device=torch.device('cuda'),
        sample_device='cuda')

In [8]:
pos = torch.tensor([1,2,3,4,5,6,7,8,9,10,11,12,13])
num_steps = len(pos)
n_envs = 100
device = 'cuda'

time_idx, env_idx = torch.meshgrid(pos, torch.arange(n_envs), indexing="ij")

In [9]:
obs = rb.obs[time_idx, env_idx]
next_obs = rb.next_obs[time_idx, env_idx]
actions = rb.actions[time_idx, env_idx]
rewards = rb.rewards[time_idx, env_idx]
dones = rb.dones[time_idx, env_idx]
associated_r = rb.associated_r[time_idx, env_idx]
q_vals = rb.Q[time_idx, env_idx]
v_vals = rb.V[time_idx, env_idx]

In [11]:
context_len = 5

In [12]:
sequences = []
for start in range(num_steps - context_len + 1):
    end = start + context_len

    # Собираем последовательности: [context_len, n_envs, ...] → [n_envs, context_len, ...]
    obs_seq = obs[start:end].to(device).permute(1, 0, *range(2, obs.ndim))
    next_obs_seq = next_obs[start:end].to(device).permute(1, 0, *range(2, next_obs.ndim))
    actions_seq = actions[start:end].to(device).permute(1, 0, *range(2, actions.ndim))
    rewards_seq = rewards[start:end].to(device).permute(1, 0)
    dones_seq = dones[start:end].to(device).permute(1, 0)
    associated_r_seq = associated_r[start:end].to(device).permute(1, 0)
    q_seq = q_vals[start:end].to(device).permute(1, 0)
    v_seq = v_vals[start:end].to(device).permute(1, 0)

    for i in range(n_envs):
        seq = (
            obs_seq[i],           # [context, ...]
            next_obs_seq[i],
            actions_seq[i],
            rewards_seq[i],
            dones_seq[i],
            associated_r_seq[i],
            q_seq[i],
            v_seq[i]
        )
        sequences.append(seq)

In [25]:
obs_seq = torch.stack([s[0] for s in sequences])
next_obs_seq = torch.stack([s[1] for s in sequences])
actions_seq = torch.stack([s[2] for s in sequences])
rewards_seq = torch.stack([s[3] for s in sequences])
dones_seq = torch.stack([s[4] for s in sequences])
associated_r_seq = torch.stack([s[5] for s in sequences]) 
q_seq = torch.stack([s[6] for s in sequences])
v_seq = torch.stack([s[7] for s in sequences])  

In [16]:
obs_seq.shape

torch.Size([900, 42])

In [26]:
dataset = TensorDataset(
            obs_seq,
            next_obs_seq,
            actions_seq,
            rewards_seq,
            dones_seq,
            associated_r_seq,
            q_seq,
            v_seq
        )
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [27]:
for batch in dataloader:
    break

In [30]:
batch[2].shape

torch.Size([64, 5, 4])